# Module 0: Connect and Verify Your Environment

This workshop is about owning the inference layer your agents run on: a model you serve yourself with [vLLM](https://docs.vllm.ai) on a dedicated [Akamai Cloud GPU](https://www.linode.com/products/gpu/), instead of renting tokens from a hosted API. Before you tune anything, you confirm the environment works. This module verifies the three things every later module depends on: your vLLM endpoint answers, your Kubernetes namespace is reachable with `kubectl`, and a real chat completion comes back from a server you control. Five minutes here saves an afternoon of debugging later.

## Learning objectives
- Resolve your connection settings from the environment and read them back
- List the pods in your namespace with `kubectl` and find your vLLM pod
- Confirm a dedicated GPU is attached to your vLLM pod's node (without cluster-wide access)
- Build an OpenAI client pointed at your own vLLM and send a chat completion
- Recognize the four environment variables every module reads

## Prerequisites
- This is the on-ramp. Start here; there is no prior module
- Path A (hosted workshop): your environment is already running, signed in with your access card
- Path B (bring your own): infrastructure stood up with the `akamai-workshop-platform` repo, variables exported
- About 5 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/) &middot; [Linode Kubernetes Engine](https://www.linode.com/products/kubernetes/) &middot; [kubectl](https://kubernetes.io/docs/reference/kubectl/) &middot; [OpenAI chat API](https://platform.openai.com/docs/api-reference/chat)

## Your environment design basics

The notebooks never provision infrastructure. They read connection details from environment variables, so the same code runs whether the environment was handed to you (Path A) or you built it yourself (Path B).

- `VLLM_HOST` is the base URL of your vLLM OpenAI-compatible endpoint, with the `/v1` suffix. Inside the cluster it resolves as a Service name like `http://vllm:8000/v1`.
- `MODEL_NAME` is the model id your server is serving. You pass it on every request.
- `NAMESPACE` is your slice of the cluster. Your kubeconfig is scoped to it, so `kubectl` only sees your own resources.
- `KUBECONFIG` points at that namespace-scoped kubeconfig file.

![Inside the Akamai LKE cluster: your JupyterLab notebooks call a vLLM Service that routes to a vLLM pod on a dedicated GPU node, all in your namespace](images/00_connect_and_verify_architecture.png)

## 1. Setup

Install the two packages this module needs to reach the endpoint. We reinstall here so this notebook stands on its own. `kubectl` is already on your PATH in the workshop environment.

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31"

## 2. Resolve your settings

`common/config.py` reads the four variables from your environment and fills in sane defaults. `print_settings()` shows the resolved values so you can see your own environment. It never prints the API key value, only whether one is set, so a screenshot stays safe to share.

In [ ]:
# Make the repo's common/ package importable from this subfolder, then print
# the settings resolved from your environment.
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from common.config import print_settings, build_client

settings = print_settings()

**What you should see:** your `VLLM_HOST`, the derived metrics URL, `MODEL_NAME`, and `NAMESPACE` printed back. On Path A these are filled in for you. If `VLLM_HOST` still shows the default `http://vllm:8000/v1` and you are on Path B outside the cluster, that name will not resolve: set the variables and restart the kernel before moving on.

## 3. Reach your namespace

`kubectl` reads your kubeconfig and talks to the cluster. Because the kubeconfig is namespace-scoped, this lists only the pods in your namespace. You should see your vLLM pod running.

In [ ]:
# Requires a live cluster (KUBECONFIG must be set).
# List the pods in your namespace. Expect your vLLM pod as Running.
ns = settings.namespace
!kubectl get pods -n {ns}

**What you should see:** a short table of pods. Your vLLM pod shows `STATUS` `Running` and `READY` `1/1`. If it is `Pending` or `ContainerCreating`, give it a minute and re-run. If `kubectl` reports a connection or permission error, your `KUBECONFIG` is not set correctly.

## 4. Confirm your GPU is attached

The model runs on a dedicated GPU. You do not run `nvidia-smi` from this notebook: the JupyterLab pod has no GPU, the vLLM pod does. Your kubeconfig is scoped to your namespace, so `kubectl get nodes` is **Forbidden by design**, and you cannot list cluster-wide resources. Instead, read your own vLLM pod: the GPU it requested, and the node it was scheduled onto. That is the GPU your inference runs on.

In [ ]:
# Requires a live cluster (in-namespace only, no cluster-wide access needed).
# Your kubeconfig is namespace-scoped, so `kubectl get nodes` is Forbidden by design.
# Instead, confirm YOUR vLLM pod requested a GPU, and see the node it landed on.
# A GPU value of 1 means a dedicated GPU is attached to that node.
ns = settings.namespace
!kubectl get pods -n {ns} -l app=vllm -o custom-columns=POD:.metadata.name,NODE:.spec.nodeName,GPU:.spec.containers[0].resources.limits.'nvidia\.com/gpu'

**What you should see:** one row for your vLLM pod with `GPU` `1` and the `NODE` it runs on. A GPU of `<none>` means the pod did not request a GPU (it would not be serving on one); an empty list means the vLLM pod is not up yet, so check step 3. On Path A this is provisioned for you.

## 5. Send your first request

This is the payoff. Build an OpenAI client pointed at your vLLM endpoint and send one chat completion. The `openai` package is the same client you would use against the hosted OpenAI API. The only change is `base_url`. That is the whole reason an OpenAI-compatible server is convenient: your existing code barely changes.

In [ ]:
# Build the client and send one chat completion through your own server.
client = build_client(settings)

resp = client.chat.completions.create(
    model=settings.model_name,
    messages=[{"role": "user", "content": "In one sentence, what is an inference server?"}],
    max_tokens=64,
    temperature=0.0,
)
print(resp.choices[0].message.content)

**What you should see:** one sentence of generated text. A connection error means `VLLM_HOST` is wrong or the pod is not ready. A 404 on the model means `MODEL_NAME` does not match what the server loaded; run `kubectl logs` on the vLLM pod to see the model id it serves.

## Things to know

- **Two paths, one notebook.** Path A hands you the environment; Path B builds it with `akamai-workshop-platform`. The code is identical because everything comes from environment variables.
- **Your kubeconfig is scoped.** `kubectl` only sees your namespace. You cannot touch a neighbor's vLLM, and they cannot touch yours.
- **The notebook pod has no GPU.** Inference runs in the vLLM pod on a GPU node. You drive and measure it over the network, which is exactly how you operate it in production.

> NOTE: The metrics URL is not a separate variable. It is derived from `VLLM_HOST`: the same host with `/metrics` instead of `/v1`. Module 2 starts reading it.

## Try it yourself

**Ask the server which models it serves.** Call the `/v1/models` endpoint directly and confirm the id matches your `MODEL_NAME`. **Stretch:** print the full models response and find the context length the server reports.

**Watch the pod come up.** Re-run the `kubectl get pods` cell a few times right after a restart and watch `READY` go from `0/1` to `1/1`.

In [ ]:
# Run as-is: ask the server which models it serves.
import requests

root = settings.vllm_host.rstrip("/").removesuffix("/v1")
data = requests.get(
    f"{root}/v1/models",
    headers={"Authorization": f"Bearer {settings.api_key}"},
    timeout=10,
).json()
ids = [m["id"] for m in data.get("data", [])]
print("served models     :", ids)
print("matches MODEL_NAME:", settings.model_name in ids)

## Summary

- The notebooks read four environment variables: `VLLM_HOST`, `MODEL_NAME`, `NAMESPACE`, `KUBECONFIG`. Everything else derives from them.
- `kubectl` against your namespace shows your vLLM pod and the GPU node it runs on.
- An OpenAI client with `base_url` set to your endpoint sends a real completion. Your code barely changes from calling a hosted API.
- You now have a verified, reachable inference server to operate for the rest of the workshop.

## Next

**Module 1: Renting vs Owning Your Inference.** You can reach the server, but you have not yet made the case for running it yourself. Next you send the same prompt to a hosted API and to your own vLLM, and look at the cost, data residency, and rate-limit differences that make owning worth it.